In [4]:
import os
import warnings
import torch
import torch.nn as nn
from transformers import AutoModel
from transformers.utils import logging as hf_logging
from huggingface_hub import logging as hf_hub_logging

# Avoid widget-based progress output in notebooks and silence non-critical load warnings.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
hf_logging.disable_progress_bar()
hf_logging.set_verbosity_error()
hf_hub_logging.set_verbosity_error()

# Silence optional HF token warnings when running without authentication.
warnings.filterwarnings(
    "ignore",
    message=r".*HF_TOKEN.*vault.*",
    category=UserWarning
)
warnings.filterwarnings(
    "ignore",
    message=r".*not authenticated with the Hugging Face Hub.*",
    category=UserWarning
)

class CodeSecurityIdentifier(nn.Module):
    def __init__(self, encoder_name="microsoft/graphcodebert-base", 
                 num_cwe_classes=68, num_extra_classes=10):
        super(CodeSecurityIdentifier, self).__init__()
        
        # Encoder backbone (without a pooler head to match GraphCodeBERT usage).
        try:
            self.encoder = AutoModel.from_pretrained(
                encoder_name,
                add_pooling_layer=False
            )
        except TypeError:
            # Fallback for model classes that do not support add_pooling_layer.
            self.encoder = AutoModel.from_pretrained(encoder_name)

        hidden_dim = self.encoder.config.hidden_size
        
        # Four heads
        self.det_head = nn.Linear(hidden_dim, 1)
        self.cwe_head = nn.Linear(hidden_dim, num_cwe_classes)
        self.severity_head = nn.Linear(hidden_dim, 4)
        self.extra_head = nn.Linear(hidden_dim, num_extra_classes)
        
    def forward(self, input_ids, attention_mask):
        # 1) Encode sequence and take CLS representation.
        encoder_outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = encoder_outputs.last_hidden_state[:, 0, :]  # [batch_size, hidden_dim]

        # 2) Connect CLS embedding to all 4 output heads.
        logits_det = self.det_head(cls_embedding).squeeze(-1)        # [batch_size]
        logits_cwe = self.cwe_head(cls_embedding)                    # [batch_size, num_cwe_classes]
        logits_severity = self.severity_head(cls_embedding)          # [batch_size, 4]
        logits_extra = self.extra_head(cls_embedding)                # [batch_size, num_extra_classes]

        # 3) Return all logits in one dictionary.
        return {
            "logits_det": logits_det,
            "logits_cwe": logits_cwe,
            "logits_severity": logits_severity,
            "logits_extra": logits_extra,
        }

In [5]:
#Example usage
import torch

#Create model instance
model = CodeSecurityIdentifier(
    encoder_name="microsoft/graphcodebert-base",
    num_cwe_classes=68,
    num_extra_classes=10
)

# Sample input data (batch_size=2)
input_ids = torch.randint(0, 1000, (2, 128))          # [batch_size, seq_length]
attention_mask = torch.ones(2, 128, dtype=torch.long) # [batch_size, seq_length]

#Forward pass
outputs = model(input_ids, attention_mask)

# Check outputs
print("Detection logits shape:", outputs["logits_det"].shape)      # [2]
print("CWE logits shape:", outputs["logits_cwe"].shape)            # [2, 68]
print("Severity logits shape:", outputs["logits_severity"].shape)  # [2, 4]
print("Extra logits shape:", outputs["logits_extra"].shape)        # [2, 10]

Detection logits shape: torch.Size([2])
CWE logits shape: torch.Size([2, 68])
Severity logits shape: torch.Size([2, 4])
Extra logits shape: torch.Size([2, 10])
